In [3]:
import muon as mu
m = mu.read(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data\Lee_EAE\all_common_DE2500_metaData.h5mu")    # Correct filename
   # or mu.read("your_file.h5mu")
# optionally remove problematic components: e.g. del m.mod['airr'] if you don't need it
m

e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
e:\Anaconda\envs\bio\Lib\site-packages\mudata\_core\mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 77500 × 2500
  obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'HTO_maxID', 'HTO_secondID', 'HTO_margin', 'HTO_classification', 'HTO_classification.global', 'hash.ID', 'percent.mt', 'integrated_nn_res.2', 'seurat_clusters', 'level_2_clusters', 'manual_cell_type', 'run_num', 'sample', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'celltype_sample', 'cloned', 'in_two_tissue', 'TCRdist_MOG'
  2 modalities
    gex:	77500 x 2500
      obs:	'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'leiden', 'modified_cell_type', 'IFN_stim', 'Activation', 'Exhaust', 'Mem', 'state', 'is_2D2'
      var:	'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'hvg', 'is_2D2_colors', 'leiden', 'leiden_colors', 'log1p', 'modified_cell_type_colors', 'mouse_id_colors', 'neighbors', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_umap'
      varm:	'PCs'
      obsp:	'connectivities', 'distances'
    airr:	77500 x 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion'
      uns:	'chain_indices', 'clone_id', 'ir_dist_nt_identity'
      obsm:	'airr', 'chain_indices'

In [4]:
# Create a combined dataframe with selected columns from different modalities
import pandas as pd

# Get columns from m.obs
obs_cols = ['VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'cloned', 'in_two_tissue']
df_obs = m.obs[obs_cols]

# Get columns from m['gex'].obs
gex_cols = ['mouse_id', 'date', 'tissue', 'sample', 'modified_cell_type', 'state']
df_gex = m['gex'].obs[gex_cols]

# Get columns from m['airr'].obs
airr_cols = ['chain_pairing', 'clone_id', 'clone_id_size']
df_airr = m['airr'].obs[airr_cols]

# Combine all dataframes with aligned indices
combined_df = pd.concat([df_obs, df_gex, df_airr], axis=1)

# Save the combined dataframe
combined_df.to_csv("combined_obs_data.csv")
print(f"Combined dataframe saved with shape: {combined_df.shape}")


Combined dataframe saved with shape: (77500, 17)


In [ ]:
import pandas as pd
import anndata as ad
import warnings
warnings.filterwarnings('ignore')

adata = ad.read_h5ad('p1_preprocessed.h5ad')
csv_data = combined_df
print(f"AnnData shape: {adata.shape}, CSV shape: {csv_data.shape}")



AnnData shape: (51500, 2500), CSV shape: (77500, 17)


In [6]:
# Merge data
common_indices = set(adata.obs.index).intersection(set(csv_data.index))
adata_merged = adata[adata.obs.index.isin(common_indices)].copy()
csv_subset = csv_data.reindex(adata_merged.obs.index)

for col in csv_subset.columns:
    adata_merged.obs[col] = csv_subset[col]

print(f"Merged {len(common_indices)} cells with {len(csv_subset.columns)} new columns")


Merged 51500 cells with 17 new columns


In [7]:
adata_merged

AnnData object with n_obs × n_vars = 51500 × 2500
    obs: 'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len', 'has_binding', 'set', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'cloned', 'in_two_tissue', 'modified_cell_type', 'state'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'hvg', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors', 'mouse_id_enc'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices', 'mouse_id_ohe'

In [8]:
# Save result
adata_merged.write('p1_preprocessed_with_metadata.h5ad')
print(f"Saved to p1_preprocessed_with_metadata.h5ad")


Saved to p1_preprocessed_with_metadata.h5ad


In [9]:
adata_merged

AnnData object with n_obs × n_vars = 51500 × 2500
    obs: 'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len', 'has_binding', 'set', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'cloned', 'in_two_tissue', 'modified_cell_type', 'state'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'hvg', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors', 'mouse_id_enc'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices', 'mouse_id_ohe'